In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch master https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile realesrgan.py realesrgan_encode.py
!cd /kaggle/working/Real-ESRGAN && python realesrgan_encode.py --help >/dev/null
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "(libx265|hevc_nvenc|libsvtav1|av1_nvenc)" || true

In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/bz.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "24000/1001"

START_TIME = 0
TEST_SECONDS = 10
PROGRESS_INTERVAL = 60.0

DENOISE_STRENGTH = 1.0
INPUT_WIDTH = 0
INPUT_HEIGHT = 0
TILE_SIZE = 0
OVERLAP = 0
BATCH_SIZE = 8
GPU_IDS = "0,1"

# 编码器：
#   CPU HEVC = "libx265"
#   GPU HEVC = "hevc_nvenc"
#   CPU AV1  = "libsvtav1"
#   GPU AV1  = "av1_nvenc"  # 需要支持 AV1 NVENC 的 NVIDIA GPU
VIDEO_CODEC = "hevc_nvenc"
CRF = 18                  # libx265 / libsvtav1；SVT-AV1 支持 0-63
PRESET = "medium"         # libx265
SVTAV1_PRESET = 6         # libsvtav1: 0-13，数值越大越快
CQ = 18                   # NVENC；AV1 NVENC 支持 0-63
NVENC_PRESET = "p7"       # p7 质量优先
ENCODE_GPU = 0

AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [ ]:
import shlex
import subprocess
import sys

command = [
    sys.executable, "/kaggle/working/Real-ESRGAN/realesrgan_encode.py",
    "--input", INPUT_VIDEO,
    "--output", OUTPUT_VIDEO,
    "--model", MODEL,
    "--model-path", MODEL_PATH,
    "--denoise-strength", str(DENOISE_STRENGTH),
    "--scale", str(SCALE),
    "--fps", str(FPS),
    "--fp16",
    "--channels-last",
    "--input-width", str(INPUT_WIDTH),
    "--input-height", str(INPUT_HEIGHT),
    "--tile-size", str(TILE_SIZE),
    "--overlap", str(OVERLAP),
    "--batch-size", str(BATCH_SIZE),
    "--gpu-ids", GPU_IDS,
    "--video-codec", VIDEO_CODEC,
    "--crf", str(CRF),
    "--preset", PRESET,
    "--svtav1-preset", str(SVTAV1_PRESET),
    "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET,
    "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC,
    "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME),
    "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg",
    "--ffprobe-bin", "ffprobe",
]

print("[command]", shlex.join(command), flush=True)
subprocess.run(command, check=True)
